# Milestone 4, Section 2.4: Active learning and human-in-the-loop simulations

Two simulations, both using the continual-learning strategy of Section 1: each candidate X-learner is retrained from scratch on the new labelled users plus a 200,000-row replay buffer (the curation simulation retrains on the whole corrected subset), and replaces the deployed model only if the paired bootstrap gate on 250,000 random-holdout users (100 resamples) gives a Qini gain whose lower bound is above zero. Both the candidate's test Qini (what the update would achieve) and the deployed model's (what the gate lets through) are reported.

1. **Active learning.** Start from 5% of the training split; add 5% per cycle for five cycles, chosen at random or with the hybrid query strategy (uncertainty |tau0 - tau1|, margin to the profitability threshold, diversity across 50 feature-space clusters; weights 0.5 / 0.3 / 0.2). Criteo's randomized outcomes play the role of the experiment results. Set `DRIFT = 2.0` to run the same comparison with all raw features shifted by +2 sd.
2. **Human label curation.** A bot attack adds fake conversions (40% on top of the real ones) to treated users of one cluster in a random 30% subset of the training split. Annotators (95% detection, 2% false flags) review 15% of the conversions per cycle for three cycles, either at random or in the clusters with the most abnormal treated-control gap.

The code is in `uplift_pipeline/hitl.py`. Run all cells (about 30-40 minutes on a Colab CPU runtime). Results are written to `outputs/hitl/`.

In [ ]:
# SETUP: install the XGBoost version the saved model was written with, and make the pipeline importable.
# The package is looked for (1) next to this notebook, (2) in an uploaded Milestone_4_code.zip,
# (3) in a clone of the GitHub repository (pulled again if it already exists).
%pip install -q xgboost==3.4.1 psutil
import os, sys, glob, zipfile, subprocess
PKG = None
if os.path.isdir("uplift_pipeline"):
    PKG = "."
if PKG is None and glob.glob("Milestone_4_code*.zip"):
    zipfile.ZipFile(sorted(glob.glob("Milestone_4_code*.zip"))[-1]).extractall(".")
    PKG = "Milestone 4"
if PKG is None:
    if os.path.isdir("INSE-6450"):
        subprocess.run(["git", "-C", "INSE-6450", "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/saopa28/INSE-6450.git"], check=True)
    PKG = os.path.join("INSE-6450", "Milestone 4")
if not os.path.isdir(os.path.join(PKG, "uplift_pipeline")):
    raise FileNotFoundError(f"uplift_pipeline not found in {PKG!r}. Push the 'Milestone 4' folder to GitHub, "
                            "or upload Milestone_4_code.zip to the Colab file panel and run this cell again.")
sys.path.insert(0, os.path.abspath(PKG))
import xgboost; print("xgboost", xgboost.__version__, "| pipeline from", os.path.abspath(PKG))

In [ ]:
import numpy as np, pandas as pd
from IPython.display import display
from uplift_pipeline import load_data, split, load_v1, OUT_DIR
from uplift_pipeline.hitl import active_learning, label_curation, plot_active_learning, plot_label_curation, AL, CURATION

FRAC, DATA = 1.0, None
DRIFT = 0.0                     # 0 = original data; 2.0 = all raw features shifted by +2 sd
OUT = OUT_DIR / "hitl"; OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 250)

df = load_data(frac=FRAC) if DATA is None else load_data(DATA, frac=FRAC)
sp = split(df); del df
v1 = load_v1()                  # only its Milestone 2 configuration is used; every model here is trained from scratch
print({k: f"{len(v[2]):,} rows" for k, v in sp.items()})
display(pd.DataFrame({"active learning": pd.Series(AL), "label curation": pd.Series(CURATION)}))

## 1. Active learning with continual updates

In [ ]:
al, al_summ = active_learning(sp, clf=v1.clf, reg=v1.reg, drift_k=DRIFT)
al.to_csv(OUT / "active_learning.csv", index=False); al_summ.to_csv(OUT / "active_learning_summary.csv")
display(al.round(6))
display(al_summ)

In [ ]:
fig = plot_active_learning(al, OUT / "active_learning.png")

## 2. Human label curation with continual updates

In [ ]:
hc, hc_summ, info = label_curation(sp, clf=v1.clf, reg=v1.reg)
hc.to_csv(OUT / "label_curation.csv", index=False); hc_summ.to_csv(OUT / "label_curation_summary.csv")
print(f"{info['fakes']:,} fake conversions added to {info['real_conversions']:,} real ones | clean-label Qini "
      f"{info['clean_qini']:.6f} | before curation {info['noisy_qini']:.6f} ({100 * info['noisy_qini'] / info['clean_qini']:.0f}% of clean)")
display(hc.round(6))
display(hc_summ)

In [ ]:
fig = plot_label_curation(hc, info, OUT / "label_curation.png")
print("Files written:", sorted(p.name for p in OUT.iterdir()))